# Tokenizers From Scratch — DA5013 Week 2

Three algorithms, implemented with nothing but the Python standard library.

**Why bother, four days before a quiz?** Because the week-2 deck hand-traces BPE across
fifteen slides and hand-traces Viterbi across eight more, and that style of question has
*not yet* appeared on a Quiz 1 paper. Writing the algorithm is the fastest way to never
get a hand-trace wrong.

Every section ends with an `assert` against a number printed on the lecture slides.
If the notebook runs clean, your implementation agrees with the course.

**Run order:** top to bottom. No `pip install` needed — `collections` and `math` only.

| Part | Builds | Verified against |
|---|---|---|
| 1 | Pre-tokenization and the starting vocabulary | 13 words, 16 occurrences, 22 symbols |
| 2 | BPE training loop | first merges `in`, then `ing` |
| 3 | Tokenizing unseen text by replaying merges | — |
| 4 | WordPiece scoring | the deck's score table |
| 5 | Unigram and Viterbi | `whereby` -> `where\|by` |
| 6 | Exam utilities: parameters and memory | every numeric PYQ answer |

In [ ]:
from collections import defaultdict
import math

# The exact corpus the week-2 lecture works through by hand.
CORPUS = ("knowing the name of something is different from knowing something. "
          "knowing something about everything isn't bad")

print(CORPUS)

---
## Part 1 — Pre-tokenization and the starting vocabulary

**Pre-tokenization** is the first rough split, on whitespace. It used to be called
"tokenization" outright; it was renamed once subword algorithms arrived, because the
real tokenization happens afterwards.

Note what whitespace splitting does *not* know about: punctuation. `something` and
`something.` become two different words, and the full stop is just another character.

In [ ]:
def word_frequencies(text):
    """Split on whitespace and count. Insertion order is preserved, which matters
    later: it is what breaks ties the same way the lecture does."""
    freq = defaultdict(int)
    for w in text.split():
        freq[w] += 1
    return dict(freq)


WF = word_frequencies(CORPUS)

for w, f in WF.items():
    print(f"{w:<12} {f}")

assert len(WF) == 13,          "expected 13 distinct words"
assert sum(WF.values()) == 16, "expected 16 total word occurrences"
print("\n13 distinct words, 16 occurrences - matches the slide")

### The starting vocabulary

Each word becomes a list of characters plus `</w>`, a symbol marking the word boundary.
That marker is what lets the tokenizer tell `ing` at the end of a word from `ing` in
the middle — the difference between *walking* and *ingredient*.

In [ ]:
def to_symbols(word_freq, end_of_word=True):
    """Each entry: (tuple_of_symbols, frequency). A tuple so it can be a dict key."""
    out = []
    for w, f in word_freq.items():
        syms = list(w) + (["</w>"] if end_of_word else [])
        out.append([syms, f])
    return out


def symbol_counts(words):
    """How many times each symbol occurs across the corpus, weighted by word frequency."""
    counts = defaultdict(int)
    for syms, f in words:
        for s in syms:
            counts[s] += f
    return dict(counts)


words = to_symbols(WF)
counts = symbol_counts(words)

print(f"starting vocabulary size: {len(counts)}")
for s in ["k", "n", "o", "i", "</w>", "'"]:
    print(f"  {s!r:<8} {counts[s]}")

assert len(counts) == 22
assert counts["k"] == 3 and counts["n"] == 13 and counts["o"] == 9
assert counts["i"] == 10 and counts["</w>"] == 16 and counts["'"] == 1
print("\nall six counts match the slide")

---
## Part 2 — BPE

Byte Pair Encoding has exactly one rule: **merge the adjacent pair you see most often,
then do it again.** Everything else is bookkeeping.

### Step 2a — count adjacent pairs

In [ ]:
def pair_counts(words):
    """Count every adjacent symbol pair. Returns (counts, first_seen_order).

    The order list exists for tie-breaking. The lecture's answer depends on it:
    (i,n) and (n,g) both occur 7 times, and (i,n) wins only because it is met first.
    """
    counts = defaultdict(int)
    order = []
    for syms, f in words:
        for a, b in zip(syms, syms[1:]):
            if (a, b) not in counts:
                order.append((a, b))
            counts[(a, b)] += f
    return dict(counts), order


pc, order = pair_counts(words)

for pair, c in sorted(pc.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {str(pair):<22} {c}")

assert pc[("i", "n")] == 7 and pc[("n", "g")] == 7
assert pc[("g", "</w>")] == 6 and pc[("t", "h")] == 5
print("\npair counts match the slide, including the 7-7 tie")

### Step 2b — apply one merge

Replace every occurrence of the winning pair with a single joined symbol. The scan is
left to right and non-overlapping: once two symbols are consumed, move past both.

In [ ]:
def merge_pair(words, pair):
    """Join `pair` wherever it appears. Mutates `words` in place and returns the new symbol."""
    a, b = pair
    joined = a + b
    for entry in words:
        syms, out, i = entry[0], [], 0
        while i < len(syms):
            if i < len(syms) - 1 and syms[i] == a and syms[i + 1] == b:
                out.append(joined)
                i += 2
            else:
                out.append(syms[i])
                i += 1
        entry[0] = out
    return joined


def pick_best(counts, order, score=None):
    """First-encountered maximum, which is what Python's max() over a dict does
    and what produces the lecture's answer on a tie."""
    key = (lambda p: counts[p]) if score is None else score
    best = order[0]
    for p in order:
        if key(p) > key(best):
            best = p
    return best


best = pick_best(pc, order)
print("winning pair:", best, "with count", pc[best])
assert best == ("i", "n"), "tie-break should favour the first-encountered pair"

### Step 2c — the training loop

That is the whole algorithm: count, merge, repeat until the vocabulary is big enough.
`num_merges` is the only hyperparameter, and it is how you dial vocabulary size.

In [ ]:
def train_bpe(word_freq, num_merges):
    """Returns (merges_in_order, final_word_forms, vocabulary)."""
    words = to_symbols(word_freq)
    merges = []
    for _ in range(num_merges):
        counts, order = pair_counts(words)
        if not counts:
            break
        best = pick_best(counts, order)
        merge_pair(words, best)
        merges.append(best)
    return merges, words, symbol_counts(words)


merges, final_words, vocab = train_bpe(WF, num_merges=6)

for i, (a, b) in enumerate(merges, 1):
    print(f"  merge {i}: ({a}, {b}) -> {a + b}")

assert merges[0] == ("i", "n"),   "first merge should be (i,n) -> in"
assert merges[1] == ("in", "g"),  "second merge should be (in,g) -> ing"
print("\nfirst two merges reproduce the slide: 'in', then 'ing'")

print("\nwords after 6 merges:")
for syms, f in final_words:
    print(f"  {' '.join(syms)}")

---
## Part 3 — Tokenizing text the tokenizer has never seen

Training produced an **ordered list of merge rules**. Tokenizing replays that list, in
order, on a fresh word. This is why BPE is called greedy and deterministic — there is no
search, just a replay.

In [ ]:
def tokenize_bpe(word, merges):
    """Split into characters, add </w>, then replay every learned merge in order."""
    syms = list(word) + ["</w>"]
    for a, b in merges:
        joined, out, i = a + b, [], 0
        while i < len(syms):
            if i < len(syms) - 1 and syms[i] == a and syms[i + 1] == b:
                out.append(joined)
                i += 2
            else:
                out.append(syms[i])
                i += 1
        syms = out
    return syms


for w in ["knowing", "everything", "unknowing", "bad"]:
    toks = tokenize_bpe(w, merges)
    print(f"  {w:<12} -> {' | '.join(toks):<34} ({len(toks)} tokens)")

assert tokenize_bpe("knowing", merges) == ["kn", "o", "w", "ing</w>"]
print("\nmatches the lab")

> **Look at `unknowing`.** The tokenizer has never seen it, yet it does not produce an
> `[UNK]`. It falls back to pieces it does know. That is the whole argument for subword
> tokenization, and it is what makes option (c) in 2026 T1 Q2 — *"they eliminate OOV by
> ensuring every word is a single token"* — exactly backwards.

### Exercise 1

Train tokenizers at several vocabulary sizes and count the tokens a fixed sentence
produces. GA2 asked this and the answer was that token count **never increases** as
vocabulary grows, and goes **flat** whenever the new merges miss your text.

In [ ]:
TEST = "knowing everything about something"

for n in [0, 2, 4, 6, 10, 20, 40]:
    ms, _, vocab = train_bpe(WF, num_merges=n)
    total = sum(len(tokenize_bpe(w, ms)) for w in TEST.split())
    print(f"  merges={n:<4} vocab={len(vocab):<4} tokens={total}")

# TODO for you: assert that the token count is non-increasing as n grows.
# Hint: collect the totals into a list and compare each with the one before.

---
## Part 4 — WordPiece

One change, and only one. Instead of raw count, rank by

$$\text{score}(a,b) = \frac{\text{count}(a,b)}{\text{count}(a)\times\text{count}(b)}$$

Dividing by the two halves penalises pairs built from already-common characters. BPE
asks *which pair do I see most?* WordPiece asks *which pair do I see more often than its
halves would predict?*

WordPiece also drops `</w>` and marks continuation pieces with `##` instead, so
`knowing` starts as `k ##n ##o ##w ##i ##n ##g`.

In [ ]:
def wordpiece_scores(words):
    """Returns (scores, first_seen_order). Character counts ignore the ## prefix,
    which is how the lecture's score table is computed."""
    counts, order = pair_counts(words)
    syms = symbol_counts(words)
    scores = {p: counts[p] / (syms[p[0]] * syms[p[1]]) for p in counts}
    return scores, order, counts, syms


wp_words = to_symbols(WF, end_of_word=False)   # no </w> for WordPiece
scores, wp_order, wp_counts, wp_syms = wordpiece_scores(wp_words)

print("the pairs the lecture tabulates:\n")
print(f"  {'pair':<14}{'count':>7}{'c(a)':>7}{'c(b)':>7}{'score':>9}")
for p in [("k","n"), ("n","o"), ("o","w"), ("i","n"), ("n","g"), ("t","h"), ("a","d")]:
    print(f"  {str(p):<14}{wp_counts[p]:>7}{wp_syms[p[0]]:>7}{wp_syms[p[1]]:>7}{scores[p]:>9.3f}")

for p, expected in [(("k","n"),0.077), (("n","o"),0.026), (("o","w"),0.111),
                    (("i","n"),0.054), (("n","g"),0.077), (("t","h"),0.125),
                    (("a","d"),0.167)]:
    assert abs(scores[p] - expected) < 0.002, f"{p} scored {scores[p]:.4f}"
print("\nevery score matches the slide to three decimals")

### The divergence — the point of the whole week

Same corpus. Same characters. Different winner.

In [ ]:
bpe_counts, bpe_order = pair_counts(to_symbols(WF))
bpe_first = pick_best(bpe_counts, bpe_order)
wp_first  = pick_best(scores, wp_order, score=lambda p: scores[p])

print(f"  BPE picks       {bpe_first}  count {bpe_counts[bpe_first]}")
print(f"  WordPiece picks {wp_first}  score {scores[wp_first]:.4f}")

print("\n  WordPiece's full top five:")
for p in sorted(scores, key=lambda p: -scores[p])[:5]:
    print(f"    {str(p):<12} count={wp_counts[p]}  "
          f"c(a)={wp_syms[p[0]]:<3} c(b)={wp_syms[p[1]]:<3} score={scores[p]:.4f}")

assert bpe_first != wp_first, "the two algorithms should diverge immediately"

> **A note on the slide.** The lecture's score table lists only the pairs it had already
> shown for BPE, and among *those* `(a,d)` at 0.167 is the winner. Over **every** pair in
> the corpus the real winner is `(r,y)` at 0.333 — `y` occurs once and `r` three times, so
> the denominator is tiny. Both facts teach the same thing, and it is the thing to carry
> into the exam:
>
> **rare halves beat common halves at equal pair count.**

---
## Part 5 — Unigram and Viterbi

Unigram runs backwards: it starts from an oversized vocabulary and deletes the subwords
it can most afford to lose. Once trained, splitting a word is a **search**, not a replay.

Each candidate segmentation has probability $P(x) = \prod_i P(x_i)$. Multiplying many
small numbers underflows, so everything is done in logarithms, where multiply becomes add
and "largest" means **closest to zero**.

Enumerating every segmentation is exponential, so we use **Viterbi**: walk left to right,
and at each position keep only the single best way of reaching it.

In [ ]:
# The vocabulary from the lecture's 'whereby' example, as log-probabilities.
LOGP = {"b": -4.70, "e": -2.70, "h": -3.34, "r": -3.36, "w": -4.29,
        "wh": -5.99, "er": -5.34, "where": -8.21, "by": -7.34,
        "he": -6.02, "ere": -6.83, "here": -7.84, "her": -7.38, "re": -6.13}


def viterbi(word, logp):
    """Best segmentation of `word`. Returns (pieces, total_log_prob, prefix_bests)."""
    n = len(word)
    best = [-math.inf] * (n + 1)
    back = [None] * (n + 1)
    best[0] = 0.0

    for i in range(1, n + 1):                 # end of the prefix
        for j in range(i):                    # where the last piece started
            piece = word[j:i]
            if piece in logp and best[j] > -math.inf:
                score = best[j] + logp[piece]
                if score > best[i]:
                    best[i], back[i] = score, (j, piece)

    pieces, i = [], n
    while i > 0 and back[i]:
        j, piece = back[i]
        pieces.append(piece)
        i = j
    return list(reversed(pieces)), best[n], best


pieces, total, prefix = viterbi("whereby", LOGP)

print(f"  best segmentation: {' | '.join(pieces)}   total log p = {total:.2f}\n")
print("  the forward pass, position by position:")
for i in range(1, len("whereby") + 1):
    shown = f"{prefix[i]:.2f}" if prefix[i] > -math.inf else "-inf"
    print(f"    {i}  {'whereby'[:i]:<10} {shown:>8}")

assert pieces == ["where", "by"]
assert abs(total - (-15.55)) < 0.01
for i, want in [(1,-4.29), (2,-5.99), (3,-8.69), (4,-11.33), (5,-8.21)]:
    assert abs(prefix[i] - want) < 0.01, f"position {i}"
print("\nevery intermediate value matches the lattice on the slide")

> **A second note on the slides.** The deck's final lattice node reads **-15.51**.
> The correct sum is $-8.21 + -7.34 = -15.55$, and every *intermediate* value on the
> slide matches this code exactly — so it is a typo in the last node, not a different
> method. Use **-15.55**.

### Why the whole word usually wins

Every extra piece multiplies in another probability below 1, so each split drags the
total further from zero. Unigram therefore prefers the **fewest** pieces its vocabulary
allows. The lecture shows this with `knowing`.

In [ ]:
# Unigram probabilities from the deck, using 16 total word occurrences as the denominator.
P = {"k": 3/16, "now": 3/16, "ing": 7/16, "know": 3/16, "knowing": 3/16}

for cand in [["k", "now", "ing"], ["know", "ing"], ["knowing"]]:
    p = math.prod(P[t] for t in cand)
    print(f"  {' | '.join(cand):<22} {p:.6f}  = {round(p * 4096):>3}/4096")

assert round(math.prod(P[t] for t in ["k","now","ing"]) * 4096) == 63
assert round(math.prod(P[t] for t in ["know","ing"])     * 4096) == 336
assert round(math.prod(P[t] for t in ["knowing"])        * 4096) == 768
print("\n63/4096, 336/4096, 768/4096 - matches the slide. Fewest pieces wins.")

### Exercise 2

Segment a word whose characters are all in the vocabulary but which has several
plausible splits. Then try one containing a character that is **not** in `LOGP` and
watch what happens — that failure mode is exactly why every real subword vocabulary
keeps all single characters.

In [ ]:
for w in ["here", "wherebyer", "hereby", "hereabouts"]:
    pieces, total, _ = viterbi(w, LOGP)
    if pieces:
        print(f"  {w:<12} -> {' | '.join(pieces):<24} {total:.2f}")
    else:
        print(f"  {w:<12} -> no segmentation exists")

# TODO for you: add every missing single character to LOGP with a very low log-prob
# (say -12.0) and re-run. Every word should now segment. Why does that guarantee it?

---
## Part 6 — Exam utilities

Two functions that answer every numeric short-answer question on the paper. Keep these;
they are also the fastest way to generate your own practice numbers.

In [ ]:
def transformer_params(V, d, L, ctx, learned_positions=True):
    """Three buckets: embeddings, attention (4d^2 per block), FFN (8d^2 per block)."""
    embeddings = V * d + (ctx * d if learned_positions else 0)
    per_block  = 4 * d * d + 8 * d * d          # 12 d^2
    return {
        "embeddings": embeddings,
        "per_block":  per_block,
        "all_blocks": L * per_block,
        "total":      embeddings + L * per_block,
    }


def training_memory_gb(n_params, bytes_per_weight=4, optimizer_states=2,
                       trainable_fraction=1.0, inference_only=False):
    """Four copies: weights, gradients, and one per optimizer state.
    The course defines 1 GB = 1e9 bytes."""
    trainable = 0.0 if inference_only else n_params * trainable_fraction
    total = (n_params * bytes_per_weight            # weights, always
             + trainable * 4                        # gradients
             + trainable * 4 * optimizer_states)    # momentum, variance
    return total / 1e9


# --- every numeric answer from both Quiz 1 papers and GA4 ---
checks = [
    ("T1 Q19  attention at d=768",      4 * 768**2,                                     2.36e6,  2e4),
    ("T1 Q20  embeddings",              transformer_params(32000,1024,1,2048)["embeddings"], 34.87e6, 2e5),
    ("T2 Q15  embeddings",              transformer_params(65000,1536,1,4096)["embeddings"], 106.1e6, 5e5),
    ("T2 Q16  attention at d=1536",     4 * 1536**2,                                    9.44e6,  5e4),
    ("GA3     ctx 512 -> 1024 at d=768",(1024-512) * 768,                               0.393e6, 2e3),
    ("GPT-1   total parameters",        transformer_params(40478,768,12,512)["total"],  116.4e6, 2e5),
]
for name, got, want, tol in checks:
    assert abs(got - want) < tol, f"{name}: got {got}"
    print(f"  {name:<34} {got/1e6:>8.2f}M")

mem = [
    ("T1 Q18  1.2B AdamW fp32", training_memory_gb(1.2e9),                      19.2),
    ("T2 Q17  5B   AdamW fp32", training_memory_gb(5e9),                        80.0),
    ("T1 Q8   4B   AdamW fp32", training_memory_gb(4e9),                        64.0),
    ("T2 Q6   7B   weights only", training_memory_gb(7e9, inference_only=True), 28.0),
    ("        7B   int8 weights", training_memory_gb(7e9, 1, inference_only=True), 7.0),
]
for name, got, want in mem:
    assert abs(got - want) < 0.1, f"{name}: got {got}"
    print(f"  {name:<34} {got:>8.1f} GB")

print("\nall sixteen checks pass")

### Exercise 3 — build your own drill

Generate random configurations, compute the answer, and test yourself before revealing it.

In [ ]:
import random

def drill(seed=None):
    rng = random.Random(seed)
    V   = rng.choice([16000, 32000, 50000, 65000, 128000])
    d   = rng.choice([768, 1024, 1536, 2048])
    ctx = rng.choice([512, 1024, 2048, 4096])
    q = (f"Vocabulary {V:,}, embedding dimension {d}, context {ctx}, "
         f"learned positional embeddings.\nTotal embedding parameters, in millions?")
    a = transformer_params(V, d, 1, ctx)["embeddings"] / 1e6
    return q, f"{a:.1f}M"


q, a = drill(seed=7)
print(q)
print("\n  ... work it out, then run the next cell")

In [ ]:
print("answer:", a)

# TODO for you: write drill_memory() on the same pattern -- pick a parameter count,
# a precision and an optimizer, and ask for the training memory in GB.

---
## What you now have

- **BPE** — count pairs, merge the most frequent, replay the merges to tokenize.
- **WordPiece** — identical, except it divides by the halves, which makes it reach for
  rare characters first.
- **Unigram** — runs backwards, and segments by Viterbi search rather than replay.
- **Exam utilities** — parameter counts and memory, checked against every published answer.

**Three sentences to carry into the quiz**

1. BPE counts, WordPiece scores, Unigram prunes.
2. Rare halves beat common halves at equal pair count.
3. Fewest pieces wins, because every extra piece multiplies in another probability below one.